# PTv3 FP16 Baseline on One GPU
Use one complete Area 5 room and a saved FP16 checkpoint. For comparison with later quantization runs, fix the room, random seed, 2 cm voxel size, and original scale, then evaluate all test fragments at that scale. The baseline PTv3 enables RPE and runs with CUDA FP16 autocast.

In [1]:
from pathlib import Path
import os
import sys
import random
import numpy as np
import torch

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "configs/s3dis/semseg-pt-v3m1-1-rpe.py").is_file())
sys.path.insert(0, str(root))
from pointcept.utils.config import Config
from pointcept.models import build_model
from pointcept.datasets import build_dataset
from pointcept.datasets.utils import collate_fn

device = torch.device("cuda:0")
print(sys.executable, torch.__version__, torch.cuda.get_device_name(device))

/home/yy/.conda/envs/pointcept-int8/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


[PointROPE] CUDA implementation unavailable (ModuleNotFoundError: No module named 'pointrope'). Using slower Pytorch fallback.


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
/home/yy/.conda/envs/pointcept-int8/bin/python 2.5.0+cu124 NVIDIA GeForce RTX 4070


In [2]:
cfg = Config.fromfile(str(root / "configs/s3dis/semseg-pt-v3m1-1-rpe.py"))
cfg.model.backbone.legacy_sparse_features = True
cfg.data.test.data_root = str(root / "data/s3dis_le_1m")
cfg.data.test.test_cfg.aug_transform = [[dict(type="RandomScale", scale=[1.0, 1.0])]]
checkpoint = root / "exp/s3dis/ptv3-area5/model/quant/model_best_fp16.pth"
if not checkpoint.exists():
    checkpoint = checkpoint.parent.parent / checkpoint.name
weights = torch.load(checkpoint, map_location="cpu", weights_only=True)["state_dict"]
assert all(v.dtype == torch.float16 for v in weights.values() if v.is_floating_point())
model = build_model(cfg.model)
model.load_state_dict({k.removeprefix("module."): v for k, v in weights.items()}, strict=True)
# On this machine, spconv implicit GEMM cannot select an FP16 algorithm.
from spconv.core import ConvAlgo
from spconv.pytorch import SubMConv3d
for layer in model.modules():
    if isinstance(layer, SubMConv3d):
        layer.algo = ConvAlgo.Native
model = model.to(device).eval()
del weights
print("Loaded", checkpoint.name, "with strict=True")

Loaded model_best_fp16.pth with strict=True


## W8A8 Latency Evaluation on the Server
This section exports `ptv3_w8a8.pth` for server evaluation. After restarting the kernel, run the first two setup cells and this section. Export calibrates the FP16 checkpoint. Linear layers use Triton INT8 dot products, and eligible sparse convolutions use spconv INT8 implicit GEMM. The same checkpoint works on RTX 4070 and RTX 3090 GPUs. The first call compiles kernels for the target GPU. The TensorRT engine cells below reproduce the earlier experiment.

In [3]:
from pointcept.models.point_transformer_v3.w8a8 import export_w8a8

def calibration_inputs():
    ds = build_dataset(cfg.data.test)
    by_name = {ds.get_data_name(i): i for i in range(len(ds))}
    for name in ["Area_5-storage_3", "Area_5-office_30", "Area_5-hallway_8"]:
        parts = ds[by_name[name]]["fragment_list"]
        for j in sorted({0, len(parts) - 1}):
            batch = collate_fn([parts[j]])
            yield {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

random.seed(25326354)
np.random.seed(25326354)
torch.manual_seed(25326354)
bundle = export_w8a8(model, calibration_inputs())
artifact = root / "exp/s3dis/ptv3-area5/model/quant/ptv3_w8a8.pth"
temporary = artifact.with_suffix(".tmp")
torch.save(bundle, temporary)
temporary.replace(artifact)
print(artifact.name, f"{artifact.stat().st_size / 2**20:.2f} MiB")
print("W8A8 Linear:", sum(v["weight"].ndim == 2 for v in bundle["layers"].values()))
print("W8A8 SubMConv3d:", sum(v["weight"].ndim == 5 for v in bundle["layers"].values()))

[2026-11-26 01:01:12,700 INFO defaults.py line 77 187321] Totally 39 x 1 samples in s3dis_le_1m Area_5 set.


ptv3_w8a8.pth 44.78 MiB
W8A8 Linear: 123
W8A8 SubMConv3d: 22


### Optional Single-Fragment Latency Comparison on the Target GPU
Reuse the exported `ptv3_w8a8.pth` by running the first two setup cells and the cell below. Both models use FP16 inputs and the same default sparse convolution strategy, with INT8 kernels for quantized layers. Warm up both models separately, then alternate timed forward passes on the same `office_34` fragment. The measurement covers only the GPU model forward pass and excludes loading, CPU preprocessing, and fragment voting. Run on an idle GPU.

In [3]:
from pointcept.models.point_transformer_v3.w8a8 import load_w8a8

@torch.inference_mode()
def compare_latency():
    seed = 25326354
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    artifact = root / "exp/s3dis/ptv3-area5/model/quant/ptv3_w8a8.pth"
    base = build_model(cfg.model).cuda().half().eval()
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)["state_dict"]
    base.load_state_dict({k.removeprefix("module."): v for k, v in state.items()}, strict=True)
    quant = load_w8a8(build_model(cfg.model).cuda(), torch.load(artifact, map_location="cpu", weights_only=True))
    ds = build_dataset(cfg.data.test)
    index = next(i for i in range(len(ds)) if ds.get_data_name(i) == "Area_5-office_34")
    fragment = ds[index]["fragment_list"][0]
    batch = {k: v.cuda() if torch.is_tensor(v) else v for k, v in collate_fn([fragment]).items()}
    batch["feat"] = batch["feat"].half()
    nets = {"FP16": base, "W8A8": quant}
    ms = {name: [] for name in nets}
    for rep in range(25):
        # Start both backends from the same RNG state in each round.
        names = list(nets) if rep % 2 == 0 else list(reversed(nets))
        for name in names:
            torch.manual_seed(seed + rep)
            start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            start.record()
            logits = nets[name](dict(batch))["seg_logits"]
            end.record()
            end.synchronize()
            if rep == 0:
                assert torch.isfinite(logits).all()
            if rep >= 5:
                ms[name].append(start.elapsed_time(end))
    print(torch.cuda.get_device_name(), len(batch["feat"]), "points")
    for name, values in ms.items():
        print(f"{name}: p50={np.median(values):.2f}, p95={np.percentile(values, 95):.2f} ms")
    print(f"W8A8 speedup (FP16 p50 / W8A8 p50): {np.median(ms['FP16'])/np.median(ms['W8A8']):.3f}x")

compare_latency()

[2026-11-26 01:05:45,419 INFO defaults.py line 77 194822] Totally 39 x 1 samples in s3dis_le_1m Area_5 set.


Can't find algo Ampere_s8s8s8s32f32tnt_m64n64k64m32n32k64A1T16832_300_C301LLL_SK_S8 in prebuilt. compile with nvrtc...


Can't find algo Ampere_s8s8s8s32f32tnt_m128n64k64m64n32k64A1T16832_200_C301LLL_SK_S8 in prebuilt. compile with nvrtc...


Can't find algo Ampere_s8s8s8s32f32tnt_m128n64k64m64n32k64A1T16832_300_C301LLL_SK_S8 in prebuilt. compile with nvrtc...


Can't find algo Ampere_s8s8s8s32f32tnt_m128n64k64m64n32k64A1T16832_400_C301LLL_SK_S8 in prebuilt. compile with nvrtc...


Can't find algo Ampere_s8s8s8s32f32tnt_m128n64k32m64n32k32A1T16816_200_C301LLL_SK_S8 in prebuilt. compile with nvrtc...


... (46 outputs left out)

Can't find algo Turing_s8s8s8s32f32tnt_m128n256k64m64n128k64A1T8816_200_C301LLL_SK_S8 in prebuilt. compile with nvrtc...


Can't find algo Turing_s8s8s8s32f32tnt_m256n128k64m128n64k64A1T8816_200_C301LLL_SK_S8 in prebuilt. compile with nvrtc...


Can't find algo Turing_s8s8s8s32f32tnt_m128n128k64m64n64k64A1T8816_200_C301LLL_SK_S8 in prebuilt. compile with nvrtc...


NVIDIA GeForce RTX 4070 228416 points
FP16: p50=219.92, p95=222.03 ms
W8A8: p50=227.26, p95=230.13 ms
W8A8 speedup (FP16 p50 / W8A8 p50): 0.968x


In [3]:
seed = 25326354
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
room = "Area_5-office_34"
dataset = build_dataset(cfg.data.test)
idx = next(i for i in range(len(dataset)) if dataset.get_data_name(i) == room)
sample = dataset[idx]
labels = torch.as_tensor(sample["segment"], device=device).long().reshape(-1)
fragments = sample["fragment_list"]
classes = cfg.data.num_classes
scores = torch.zeros((labels.numel(), classes), device=device)
votes = torch.zeros_like(labels)
# Start each subsequent backend with the same RNG state for serialization.
eval_cpu_rng = torch.get_rng_state()
eval_cuda_rng = torch.cuda.get_rng_state(device)
with torch.inference_mode():
    for fragment in fragments:
        batch = collate_fn([fragment])
        batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(batch)["seg_logits"]
        indices = batch["index"].long()
        scores.index_add_(0, indices, logits.float().softmax(dim=-1))
        votes.index_add_(0, indices, torch.ones_like(indices))

assert torch.all(votes > 0) and torch.isfinite(scores).all()
pred = scores.argmax(dim=1)
valid = (labels >= 0) & (labels < classes)
cm = torch.bincount(labels[valid] * classes + pred[valid], minlength=classes**2).reshape(classes, classes)
intersection = cm.diag()
union = cm.sum(0) + cm.sum(1) - intersection
print(f"{room} | points={labels.numel():,} | fragments={len(fragments)} | logits={logits.dtype}")
print(f"Point accuracy={intersection.sum().item()/cm.sum().item():.4%}")
print(f"Room mIoU (nonempty union)={(intersection[union > 0] / union[union > 0]).mean().item():.4%}")

[2026-11-25 20:22:27,358 INFO defaults.py line 77 20867] Totally 39 x 1 samples in s3dis_le_1m Area_5 set.


Area_5-office_34 | points=889,970 | fragments=22 | logits=torch.float16
Point accuracy=92.3465%
Room mIoU (nonempty union)=85.0389%


## INT8 Weight Storage and Linear-Layer W8A8
Run the cells in order from a fresh `pointcept-int8` kernel. The baseline uses an FP16 checkpoint and autocast. The default `all_weights` mode stores all trainable floating-point parameters in INT8, including sparse convolution weights, RPE, biases, and normalization parameters. At runtime, all linear layers are candidates for TensorRT W8A8 subgraphs. Other parameters are decoded and used by their original operators. TensorRT selects the execution precision for each engine. Set `scope` to `mlp` to reproduce the earlier run. Calibration and evaluation use separate rooms. Restart the kernel before recalibrating to avoid quantizing converted modules twice.

This example targets Linux x86-64 with CUDA 12 and TensorRT 10.9. It requires `nvidia-modelopt[torch]==0.27.0`, `onnx==1.17.0`, and `tensorrt-cu12==10.9.0.34`. TensorRT is imported when the engines are built.

With the `pointcept-int8` environment active, install the PyTorch quantization dependencies and verify the imports:

```bash
python -m pip install "nvidia-modelopt[torch]==0.27.0" "torchprofile==0.0.4"
python -c "import modelopt.torch.quantization as mtq; print('ModelOpt PTQ import OK')"
```

In [ ]:
from importlib.metadata import version
import modelopt.torch.quantization as mtq
from pointcept.models.point_transformer_v3.point_transformer_v3m1_base import MLP

print("ModelOpt", version("nvidia-modelopt"))
calibration_rooms = ["Area_5-office_30", "Area_5-storage_3", "Area_5-hallway_8"]
assert room not in calibration_rooms
indices_by_room = {dataset.get_data_name(i): i for i in range(len(dataset))}
scope = "all_weights"  # Use "mlp" to reproduce the earlier run.
assert scope in {"mlp", "all_weights"}
mlps = {name: m for name, m in model.named_modules() if isinstance(m, MLP)}
linears = {name: m for name, m in model.named_modules() if type(m) is torch.nn.Linear}
assert mlps and all(type(m.fc1) is torch.nn.Linear for m in mlps.values()), "Restart the kernel"
units = dict(mlps)
standalone = {}
if scope == "all_weights":
    mlp_prefix = tuple(name + "." for name in mlps)
    standalone = {k: m for k, m in linears.items() if not k.startswith(mlp_prefix)}
    # Wrap each linear layer so conversion and the forward pass use the same instance.
    units.update({k: torch.nn.Sequential(m) for k, m in standalone.items()})
covered = {k for k in linears if k in units or k.startswith(tuple(x + "." for x in mlps))}
if scope == "all_weights":
    assert covered == set(linears)
print(f"{scope}: {len(covered)}/{len(linears)} Linear; {len(units)} engine subgraphs")


In [ ]:
def gpu_batch(fragment):
    batch = collate_fn([fragment])
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

def room_metrics(matrix):
    diag = matrix.diag()
    den = matrix.sum(0) + matrix.sum(1) - diag
    return (100 * diag.sum().item() / matrix.sum().item(),
            100 * (diag[den > 0] / den[den > 0]).mean().item())

@torch.inference_mode()
def evaluate_room(net):
    torch.set_rng_state(eval_cpu_rng)
    torch.cuda.set_rng_state(eval_cuda_rng, device)
    total = torch.zeros_like(scores)
    coverage = torch.zeros_like(votes)
    for part in fragments:
        inputs = gpu_batch(part)
        index = inputs["index"].long()
        with torch.autocast("cuda", dtype=torch.float16):
            out = net(inputs)["seg_logits"]
        total.index_add_(0, index, out.float().softmax(dim=-1))
        coverage.index_add_(0, index, torch.ones_like(index))
    assert coverage.min() > 0 and torch.isfinite(total).all()
    hist = torch.bincount(labels[valid] * classes + total.argmax(1)[valid],
                          minlength=classes**2).reshape(classes, classes)
    return room_metrics(hist)


baseline_metrics = room_metrics(cm)
results = {"FP16": baseline_metrics}

In [ ]:
import tempfile
run_dir = Path(tempfile.mkdtemp(prefix=f"trt-int8-{scope}-", dir=checkpoint.parent))
source = torch.load(checkpoint, map_location="cpu", weights_only=True)["state_dict"]
source = {k.removeprefix("module."): v for k, v in source.items()}
param_names = set(dict(model.named_parameters()))

def pack_weights(state):
    values, scales = {}, {}
    for name, tensor in state.items():
        if name not in param_names or not tensor.is_floating_point():
            values[name] = tensor
            continue
        x = tensor.float()
        # Use per-output-channel quantization for KRSC kernels and linear weights, and per-tensor quantization for 1D biases and normalization parameters.
        maximum = (x.abs().amax(dim=tuple(range(1, x.ndim)), keepdim=True)
                   if x.ndim > 1 else x.abs().max())
        scale = torch.where(maximum == 0, 1.0, maximum / 127)
        scales[name] = scale
        values[name] = (x / scale).round().clamp(-127, 127).to(torch.int8)
    return {"values": values, "scales": scales}

def unpack_weights(packed):
    return {k: v.float() * packed["scales"][k] if k in packed["scales"] else v
            for k, v in packed["values"].items()}

if scope == "all_weights":
    archive = run_dir / "all_weights_int8.pth"
    torch.save(pack_weights(source), archive)
    packed = torch.load(archive, map_location="cpu", weights_only=True)
    assert set(packed["scales"]) == param_names
    decoded = unpack_weights(packed)
    model.load_state_dict(decoded, strict=True)
    results["INT8 weight-only"] = evaluate_room(model)
    print(f"INT8 checkpoint: {archive.stat().st_size / 2**20:.2f} MiB; "
          f"FP16 checkpoint: {checkpoint.stat().st_size / 2**20:.2f} MiB")
    print("INT8 weight-only accuracy (%), room mIoU (%)", results["INT8 weight-only"])
    # TensorRT quantizes the original linear weights. Other layers use the decoded values from the archive.
    dense_prefix = tuple(name + "." for name in covered)
    for k in decoded:
        if k.startswith(dense_prefix):
            decoded[k] = source[k]
    model.load_state_dict(decoded, strict=True)
    del packed, decoded


### W8A8 Calibration and Accuracy Comparison on the Same Evaluation Room
Use separate rooms for calibration and enable only the selected linear-layer quantizers.

In [ ]:
@torch.no_grad()
def calibrate_blocks(converted):
    for (name, _), unit in zip(units.items(), converted):
        if name in standalone:
            parent, _, child = name.rpartition(".")
            setattr(model.get_submodule(parent), child, unit[0])
    for name in calibration_rooms:
        parts = dataset[indices_by_room[name]]["fragment_list"]
        # Voxel fragments from the same room overlap heavily, so take the first and last fragments in the first round.
        for j in sorted({0, len(parts) - 1}):
            with torch.autocast("cuda", dtype=torch.float16):
                model(gpu_batch(parts[j]))
        print("Calibrated", name)


targets = torch.nn.ModuleList(list(units.values())).eval()
mtq.quantize(targets, mtq.INT8_DEFAULT_CFG, forward_loop=calibrate_blocks)
for k in covered:
    m = model.get_submodule(k)
    for q in (m.weight_quantizer, m.input_quantizer):
        assert q.amax is not None and torch.isfinite(q.amax).all(), k
results["ModelOpt W8A8"] = evaluate_room(model)
print("Point accuracy (%), room mIoU (%)", results)


### Q/DQ Export and TensorRT Engine Build
Each block has its own weights. Save the engine plans and remaining FP16 weights in a new experiment directory next to the checkpoint. The MLP processes points independently, so point sequences above the profile limit can be split into chunks. Intermediate ONNX graphs are used only to build the engines.

In [ ]:
import io
import tempfile
import onnx
import tensorrt as trt

assert trt.__version__.startswith("10."), f"Expected TensorRT 10.x, found {trt.__version__}"
logger = trt.Logger(trt.Logger.WARNING)
max_rows = 8192

def build_engine(mlp, path):
    width = mlp.fc1.in_features if isinstance(mlp, MLP) else mlp[0].in_features
    example = torch.zeros((64, width), device=device, dtype=torch.float32)
    buf = io.BytesIO()
    with torch.no_grad():
        torch.onnx.export(mlp.eval(), example, buf, opset_version=17,
                          input_names=["x"], output_names=["y"],
                          dynamic_axes={"x": {0: "points"}, "y": {0: "points"}})
    graph = buf.getvalue()
    ops = {node.op_type for node in onnx.load_model_from_string(graph).graph.node}
    assert {"QuantizeLinear", "DequantizeLinear"} <= ops, f"Missing Q/DQ in {path.name}"
    builder = trt.Builder(logger)
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
    parser = trt.OnnxParser(network, logger)
    if not parser.parse(graph):
        raise RuntimeError([str(parser.get_error(i)) for i in range(parser.num_errors)])
    config = builder.create_builder_config()
    config.set_flag(trt.BuilderFlag.FP16)
    config.set_flag(trt.BuilderFlag.INT8)
    config.profiling_verbosity = trt.ProfilingVerbosity.DETAILED
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 256 << 20)
    profile = builder.create_optimization_profile()
    profile.set_shape("x", (1, width), (1024, width), (max_rows, width))
    config.add_optimization_profile(profile)
    plan = builder.build_serialized_network(network, config)
    if plan is None:
        raise RuntimeError(f"TensorRT could not build {path.name}")
    path.write_bytes(bytes(plan))

for name, mlp in units.items():
    build_engine(mlp, run_dir / f"{name}.plan")
    print("Built", name, flush=True)
print(run_dir)

In [ ]:
# Evaluate on one GPU with lower memory use by reusing existing plans and keeping only one engine and context on the GPU at a time.
import gc

torch.cuda.synchronize(device)
for old in list(model.modules()):
    if type(old).__name__ == "TensorRTMLP":
        if hasattr(old, "context"):
            old.context = None
        if hasattr(old, "engine"):
            old.engine = None
gc.collect()
torch.cuda.empty_cache()

class TensorRTMLP(torch.nn.Module):
    def __init__(self, runtime, path):
        super().__init__()
        self.runtime = runtime
        self.path = path

    def forward(self, x):
        stream = torch.cuda.current_stream(x.device)
        engine = self.runtime.deserialize_cuda_engine(self.path.read_bytes())
        if engine is None:
            raise RuntimeError(f"Cannot deserialize {self.path.name}")
        ctx = engine.create_execution_context()
        if ctx is None:
            free, total = torch.cuda.mem_get_info(x.device)
            raise RuntimeError(f"TensorRT context creation failed for {self.path.name}; "
                               f"free GPU memory {free / 2**20:.0f}/{total / 2**20:.0f} MiB")
        try:
            assert all(engine.get_tensor_dtype(k) == trt.float32 for k in ("x", "y"))
            outputs = []
            for chunk in x.split(max_rows):
                inp = chunk.float().contiguous()
                if not ctx.set_input_shape("x", tuple(inp.shape)):
                    raise RuntimeError(f"Shape outside profile: {inp.shape}")
                out = torch.empty(tuple(ctx.get_tensor_shape("y")), device=x.device,
                                  dtype=torch.float32)
                ctx.set_tensor_address("x", inp.data_ptr())
                ctx.set_tensor_address("y", out.data_ptr())
                if not ctx.execute_async_v3(stream.cuda_stream):
                    raise RuntimeError(f"TensorRT enqueue failed for {self.path.name}")
                outputs.append(out.to(torch.float16))
            return outputs[0] if len(outputs) == 1 else torch.cat(outputs, dim=0)
        finally:
            stream.synchronize()
            del ctx, engine

runtime = trt.Runtime(logger)
engines = {name: TensorRTMLP(runtime, run_dir / f"{name}.plan") for name in units}
for name, engine in engines.items():
    parent, _, child = name.rpartition(".")
    setattr(model.get_submodule(parent), child, engine)
results["TensorRT W8A8"] = evaluate_room(model)
for label, (acc, miou) in results.items():
    print(f"{label:18} accuracy={acc:.4f}%  room mIoU={miou:.4f}%  Δ={miou-results['FP16'][1]:+.4f} pp")

# Extract the retained weights from the original FP16 file to avoid counting the MLP and floating-point master copies twice.
prefixes = tuple(name + "." for name in units)
rest = {k: v for k, v in source.items() if not k.startswith(prefixes)}
assert 0 < len(rest) < len(source)
if scope == "all_weights":
    rest_path = run_dir / "remaining_packed.pth"
    torch.save(pack_weights(rest), rest_path)
else:
    rest_path = run_dir / "remaining_fp16.pth"
    torch.save(rest, rest_path)
plans = list(run_dir.glob("*.plan"))
assert len(plans) == len(units)
plan_bytes = sum(p.stat().st_size for p in plans)
total_bytes = rest_path.stat().st_size + plan_bytes
print(f"FP16 checkpoint: {checkpoint.stat().st_size / 2**20:.2f} MiB")
print(f"{len(plans)} plans: {plan_bytes / 2**20:.2f} MiB; "
      f"remaining weights: {rest_path.stat().st_size / 2**20:.2f} MiB")
print(f"TensorRT deployment: {total_bytes / 2**20:.2f} MiB")
print("Deployment files:", run_dir)
